# GTM Agent — Pipeline Notebook

Operator interface for running the GTM lead research pipeline.

**Flow:**
1. Configure inputs (website, ICP description, optional customer list)
2. Run pipeline → review ICP → confirm
3. Review ranked companies
4. Export to CSV

**Requirements:** Set `HUNTER_API_KEY` and `ANTHROPIC_API_KEY` in `.env` before running.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

from src.pipeline import run_pipeline, export_to_csv

## 1 — Configure Inputs

Edit the variables below before running.

In [ ]:
# ── Required ──────────────────────────────────────────────────────────────
company_website = "https://example.com"

target_description = """
Financial services and insurance companies in North America with 100-2000 employees
that need to automate their compliance and operations documentation.
"""

# ── Optional ──────────────────────────────────────────────────────────────
existing_customers = [
    # {"domain": "stripe.com", "name": "Stripe"},
]

lead_list = None  # Set to [{"domain": "..."}] for rank-only mode (Mode B)

competitors = [
    # "competitor.com",
]

list_size = 20

## 2 — Run Pipeline

In [ ]:
session = await run_pipeline(
    company_website=company_website,
    target_description=target_description,
    existing_customers=existing_customers or None,
    lead_list=lead_list,
    competitors=competitors or None,
    list_size=list_size,
)

print(f"Status: {session.status}")
print(f"Candidates found: {len(session.candidate_companies or [])}")
print(f"Ranked companies: {len(session.ranked_companies or [])}")

## 3 — Review ICP

Check the system's interpretation of your ICP. If anything looks wrong, adjust `target_description` above and re-run.

In [ ]:
icp = session.icp_definition
print(icp.model_dump_json(indent=2))

if icp.warnings:
    print("\n⚠️  Warnings:")
    for w in icp.warnings:
        print(f"  - {w}")

In [ ]:
# Confirm ICP looks correct before reviewing companies
session.icp_confirmed = True
print("ICP confirmed.")

## 4 — Review Ranked Companies

In [ ]:
ranked = session.ranked_companies or []

tier_counts = {}
for rc in ranked:
    tier_counts[rc.tier] = tier_counts.get(rc.tier, 0) + 1

print("Tier distribution:")
for tier, count in sorted(tier_counts.items()):
    print(f"  {tier}: {count}")
print()

In [ ]:
print("=" * 70)
for rc in ranked:
    c = rc.company
    contacts = rc.contacts or []
    print(f"{rc.tier} | {rc.total_score:.0f} | {c.name} ({c.domain})")
    print(f"  Industry: {c.industry or 'n/a'}  |  Size: {c.employee_range or 'n/a'}  |  Country: {c.hq_country or 'n/a'}")
    print(f"  Why now: {rc.reasoning_summary}")
    if contacts:
        for contact in contacts:
            email_str = f" <{contact.email}>" if contact.email else ""
            print(f"  → {contact.full_name}, {contact.title}{email_str}")
    print()

## 5 — Export to CSV

In [ ]:
import os
from datetime import date

os.makedirs('../output', exist_ok=True)
output_path = f"../output/leads_{date.today().isoformat()}.csv"

result_path = export_to_csv(session, output_path)
print(f"Exported to: {result_path}")